# Fraud Decision Feedback Loop

**An executable learning plane for delayed fraud outcomes**

This notebook follows a transaction after the real-time gateway has made its decision. It persists the exact feature view and model/policy versions, waits for the outcome to become observable, performs an as-of join, and monitors policy quality and drift.

**Evidence boundary.** `Amount`, `Time`, anonymized `V1…V28`, and `Class` come from OpenML dataset 1597. Calendar timestamps, risk scores, actions, entity tokens, and label delays are deterministic simulations. No bank-performance claim is made.

## 1 · Load and verify the observed data

The extract keeps all 492 fraud rows and a deterministic sample of legitimate rows. `sampling_weight` restores the legitimate-class contribution for population-level summaries; raw extract prevalence is deliberately not reported as the source prevalence.

In [1]:
from pathlib import Path
import json, sys
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / "feedback_store.py").exists())
sys.path.insert(0, str(ROOT))
from feedback_store import (
    DecisionEvent, LabelEvent, append_decisions, append_labels, connect,
    point_in_time_training_rows, policy_metrics, population_stability_index,
)

DATA = ROOT / "data" / "creditcard_openml_1597_extract.csv.gz"
MANIFEST = ROOT / "data" / "creditcard_openml_1597_manifest.json"
df = pd.read_csv(DATA).sort_values(["Time", "_source_row_number"], kind="stable").reset_index(drop=True)
manifest = json.loads(MANIFEST.read_text())

PALETTE = {"ink":"#203047", "blue":"#6F9FD8", "teal":"#75B5A7", "amber":"#D6A65D", "rose":"#C97A86", "paper":"#F7F8FA", "grid":"#E5E9EF"}
def finish(fig, title, subtitle=""):
    fig.update_layout(
        title={"text": f"<b>{title}</b><br><sup>{subtitle}</sup>", "x": 0.02},
        template="plotly_white", paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["paper"],
        font={"family": "Inter, Avenir, Arial", "color": PALETTE["ink"]},
        margin={"l":50,"r":30,"t":90,"b":50}, height=430,
        legend={"orientation":"h","y":1.04,"x":0.55},
    )
    fig.update_xaxes(gridcolor=PALETTE["grid"], zeroline=False)
    fig.update_yaxes(gridcolor=PALETTE["grid"], zeroline=False)
    return fig

pd.DataFrame({
    "source_rows":[manifest["source_rows"]], "source_fraud_rows":[manifest["source_fraud_rows"]],
    "extract_rows":[len(df)], "extract_fraud_rows":[int(df.Class.sum())],
    "source_fraud_prevalence":[manifest["source_fraud_prevalence"]],
    "extract_sha256":[manifest["extract_sha256"][:16] + "…"],
})

,source_rows,source_fraud_rows,extract_rows,extract_fraud_rows,source_fraud_prevalence,extract_sha256
0,284807,492,3000,492,0.001727,cf7398ad15c1a99e…


## 2 · Generate a versioned decision stream

A chronological split prevents later rows from influencing the model used for earlier decisions. The small logistic model is an inspectable scoring surrogate; the focus here is event lineage and label maturity, not leaderboard performance.

In [2]:
features = [f"V{i}" for i in range(1, 29)] + ["Amount"]
cut = int(len(df) * 0.70)
train, live = df.iloc[:cut].copy(), df.iloc[cut:].copy()
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2_000, random_state=42))
model.fit(train[features], train.Class)
live["probability"] = model.predict_proba(live[features])[:, 1]
live["risk_score"] = (100 * live.probability).clip(0, 100)
live["action"] = np.select([live.risk_score > 75, live.risk_score >= 40], ["HARD_BLOCK", "STEP_UP"], default="APPROVE")

pd.DataFrame({"metric":["PR-AUC", "ROC-AUC"], "value":[
    average_precision_score(live.Class, live.probability, sample_weight=live.sampling_weight),
    roc_auc_score(live.Class, live.probability, sample_weight=live.sampling_weight),
]})

,metric,value
0,PR-AUC,0.612231
1,ROC-AUC,0.974672


In [3]:
fig = go.Figure()
for label, color in [(0, PALETTE["blue"]), (1, PALETTE["rose"])]:
    fig.add_trace(go.Histogram(x=live.loc[live.Class == label, "risk_score"], nbinsx=35, histnorm="probability", opacity=0.72, name="Fraud" if label else "Legitimate", marker_color=color))
fig.update_layout(barmode="overlay")
fig.update_xaxes(title="Risk score")
fig.update_yaxes(title="Share within class")
finish(fig, "Live-window score separation", "Chronological holdout · model score is simulated; source labels are observed")
fig.show()

## 3 · Persist the exact decision-time state

The warehouse stores the score, action, model and policy versions, feature snapshot, and digest as one immutable event. Replaying the same decision is idempotent; conflicting IDs fail at the storage boundary.

In [4]:
anchor = datetime(2025, 1, 1, tzinfo=timezone.utc)
live["event_ts"] = [anchor + timedelta(seconds=float(value - live.Time.min())) for value in live.Time]

decisions, labels = [], []
for _, row in live.iterrows():
    source_row = int(row["_source_row_number"])
    decision_id = f"dec-{source_row:07d}"
    event = DecisionEvent(
        decision_id=decision_id, event_ts=row.event_ts,
        entity_ref=f"entity-{source_row % 211:04d}", amount=float(row.Amount),
        risk_score=float(row.risk_score), action=str(row.action),
        model_version="logistic-surrogate-1.0", policy_version="three-way-router-1.0",
        features={key: float(row[key]) for key in ["V4", "V10", "V14", "V17", "Amount"]},
    )
    decisions.append(event)
    labels.append(LabelEvent(
        label_id=f"lab-{source_row:07d}", decision_id=decision_id,
        observed_ts=row.event_ts + timedelta(days=30 + source_row % 61),
        is_fraud=bool(row.Class), source="simulated_maturity",
    ))

db = connect()
inserted_decisions = append_decisions(db, decisions)
inserted_labels = append_labels(db, labels)
replayed = append_decisions(db, decisions[:25])
pd.DataFrame({"decision_events":[inserted_decisions], "label_events":[inserted_labels], "duplicate_rows_inserted":[replayed], "first_audit_digest":[decisions[0].audit_digest[:20] + "…"]})

,decision_events,label_events,duplicate_rows_inserted,first_audit_digest
0,900,900,0,787cac70b253abe15d6f…


## 4 · Enforce label maturity with an as-of join

Chargebacks and investigator outcomes arrive after authorization. A query run on day 45 must not see a label that arrives on day 60. This is the central leakage control in the feedback plane.

In [5]:
last_decision = max(event.event_ts for event in decisions)
checkpoints = [last_decision + timedelta(days=day) for day in (15, 30, 45, 60, 75, 90)]
curve = []
for checkpoint in checkpoints:
    rows = point_in_time_training_rows(db, checkpoint)
    curve.append({"as_of":checkpoint, "matured_rows":len(rows), "matured_fraud":sum(int(row["is_fraud"]) for row in rows)})
maturity = pd.DataFrame(curve)

fig = go.Figure()
fig.add_trace(go.Scatter(x=maturity.as_of, y=maturity.matured_rows, mode="lines+markers", name="All matured", line={"color":PALETTE["blue"],"width":3}))
fig.add_trace(go.Scatter(x=maturity.as_of, y=maturity.matured_fraud, mode="lines+markers", name="Matured fraud", line={"color":PALETTE["rose"],"width":3}))
fig.update_xaxes(title="Evaluation date")
fig.update_yaxes(title="Labels observable by date")
finish(fig, "Delayed labels become usable gradually", "Every point is produced by the same point-in-time SQL join")
fig.show()

## 5 · Monitor policy quality only on mature outcomes

Precision and recall below describe the simulated three-way policy on the observed labels available at each checkpoint. They are demonstration metrics, not production claims.

In [6]:
monitoring = []
for checkpoint in checkpoints:
    rows = point_in_time_training_rows(db, checkpoint)
    if rows:
        monitoring.append({"as_of":checkpoint, **policy_metrics(rows)})
metrics = pd.DataFrame(monitoring)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=metrics.as_of, y=metrics.fraud_recall, name="Fraud recall", line={"color":PALETTE["rose"],"width":3}, mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Scatter(x=metrics.as_of, y=metrics.block_precision, name="Block precision", line={"color":PALETTE["teal"],"width":3}, mode="lines+markers"), secondary_y=False)
fig.add_trace(go.Bar(x=metrics.as_of, y=metrics.matured_rows, name="Matured labels", marker_color=PALETTE["amber"], opacity=0.28), secondary_y=True)
fig.update_yaxes(title="Policy metric", range=[0,1], tickformat=".0%", secondary_y=False)
fig.update_yaxes(title="Matured rows", secondary_y=True)
finish(fig, "Policy evidence changes as labels mature", "Report the maturity date with every fraud-performance estimate")
fig.show()
metrics.round(4)

,as_of,matured_rows,fraud_prevalence,block_rate,block_precision,fraud_recall
0,2025-01-31 11:23:58+00:00,8.0,0.1250,0.1250,1.0000,1.0000
1,2025-02-15 11:23:58+00:00,225.0,0.1244,0.1156,0.9615,0.8929
2,2025-03-02 11:23:58+00:00,451.0,0.1109,0.1064,0.9375,0.9000
3,2025-03-17 11:23:58+00:00,683.0,0.1171,0.1113,0.9211,0.8750
4,2025-04-01 11:23:58+00:00,900.0,0.1222,0.1144,0.9320,0.8727


## 6 · Detect score drift before promotion

PSI is a monitoring trigger, not proof of concept drift or model failure. It tells an operator where to investigate. Promotion still requires label-aware performance, calibration, queue-capacity, and fairness checks.

In [7]:
mid = len(live) // 2
reference_scores = live.risk_score.iloc[:mid].tolist()
current_scores = live.risk_score.iloc[mid:].tolist()
psi = population_stability_index(reference_scores, current_scores)

fig = go.Figure()
fig.add_trace(go.Violin(y=reference_scores, name="Earlier live window", box_visible=True, meanline_visible=True, line_color=PALETTE["blue"]))
fig.add_trace(go.Violin(y=current_scores, name="Later live window", box_visible=True, meanline_visible=True, line_color=PALETTE["amber"]))
fig.update_yaxes(title="Risk score")
finish(fig, "Score distribution monitoring", f"PSI = {psi:.3f} · investigate ≥ 0.10; do not auto-retrain from PSI alone")
fig.show()
{"score_psi":round(psi, 4), "monitoring_state":"investigate" if psi >= 0.10 else "stable"}

{'score_psi': 0.1354, 'monitoring_state': 'investigate'}

## 7 · Promotion gate and operating conclusion

1. **Shadow:** challenger consumes the same frozen feature snapshot and cannot affect settlement.
2. **Maturity wait:** compare champion and challenger only on labels observable by the evaluation date.
3. **Canary:** route a bounded segment after latency, drift, queue-load, and policy-value gates pass.
4. **Rollback:** pin model and policy versions independently; preserve every decision envelope.

The useful agentic layer is off the payment path: agents may assemble evidence, compare policy versions, and draft an investigator brief. They cannot approve funds, alter thresholds, promote a model, or overwrite the authoritative event. This keeps the control plane assistive and auditable while the decision plane remains bounded and deterministic.